# Location-aware training

This notebook trains a CNN with recording-level labels and evaluates its patient-level performance.

In [5]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))

from src.evaluation.aggregate_patients import aggregate_patients, metrics
from src.preprocessing.create_indexes import create_indexes
from src.preprocessing.split_patients import split_patients
from src.models.train_cnn import main as train_cnn

create_indexes()
split_patients()
recordings = pd.read_csv(root / 'data/processed/recording_splits.csv', dtype={'patient_id': str})
experiment_dir = root / 'results'
experiment_dir.mkdir(parents=True, exist_ok=True)


Patients: 942
Recordings: 3163

Patients per murmur label:
murmur_label
Absent     695
Present    179
Unknown     68
Name: count, dtype: int64

Indexes saved to: /Users/mikkel/Documents/GitHub/heart-murmur-detection/data/processed
Patients per split and murmur label:
murmur_label  Absent  Present  Unknown
split                                 
test             105       27       10
train            486      125       48
validation       104       27       10

Recordings per split:
split
train         2219
test           475
validation     469
Name: count, dtype: int64

Saved patient_splits.csv and recording_splits.csv


## Label audit

We counts patient and recording labels before model training.

In [6]:
recording_counts = recordings.groupby(['patient_label', 'recording_label']).size().unstack(fill_value=0)
print(recording_counts)
print('Changed Present recordings:', int(((recordings.patient_label == 'Present') & (recordings.recording_label == 'Absent')).sum()))
print('Patient split and label checks passed.')

recording_label  Absent  Present  Unknown
patient_label                            
Absent             2391        0        0
Present             117      499        0
Unknown               0        0      156
Changed Present recordings: 117
Patient split and label checks passed.


## Location-aware CNN

We train the CNN on recording-level labels and selects the best trial by patient-level validation macro-F1.

In [7]:
train_cnn(label_mode='recording', selection_mode='patient', output_dir=experiment_dir, output_prefix='location_aware_cnn', save_diagnostics=False)

GPUs found: none (using CPU)
Building features for 3163 recordings...


/opt/anaconda3/envs/my_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved feature cache to /Users/mikkel/Documents/GitHub/heart-murmur-detection/data/processed/model_cache_location_aware
Windows: 18831 | RF features: 56 | log-Mel shape: (64, 126)
true_label  Present  Absent  Unknown
split                               
train          2627   10039      671
val             486    2092      132
test            465    2211      108
Class weights: {'Present': 2.02, 'Absent': 0.42, 'Unknown': 6.63}

=== Trial 1/5: {'learning_rate': 0.001, 'batch_size': 32, 'filters': (16, 32, 64), 'dropout': 0.3} ===
Epoch 1/40


E0000 00:00:1795308856.983994 28932954 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
E0000 00:00:1795308876.060040 28932954 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type

417/417 - 22s - 53ms/step - accuracy: 0.3818 - loss: 1.0820 - val_accuracy: 0.4982 - val_loss: 1.0085 - val_patient_macro_f1: 0.2908
Epoch 2/40


E0000 00:00:1795308896.658046 28932954 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


... (135 outputs left out)

E0000 00:00:1795310500.943521 28932954 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


417/417 - 19s - 45ms/step - accuracy: 0.5947 - loss: 0.7695 - val_accuracy: 0.5827 - val_loss: 0.8134 - val_patient_macro_f1: 0.6189
Trial 5: val macro-F1 0.6888 (best epoch 6, 245 s)

Writing predictions for all splits...


E0000 00:00:1795310509.220507 28932954 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}



 trial  learning_rate  batch_size       filters  dropout  epochs_run  best_epoch  val_macro_f1 selection
     1         0.0010          32  (16, 32, 64)      0.3           8           2      0.695419   patient
     2         0.0003          32  (16, 32, 64)      0.3          14           8      0.657252   patient
     3         0.0010          64  (16, 32, 64)      0.3          25          19      0.775350   patient
     4         0.0010          32 (32, 64, 128)      0.3          11           5      0.696223   patient
     5         0.0010          32  (16, 32, 64)      0.5          12           6      0.688764   patient

Best trial: 3
CNN validation macro-F1 (patient level): 0.7754
Saved outputs to /Users/mikkel/Documents/GitHub/heart-murmur-detection/results


## Test evaluation

We aggregate the selected predictions by patient and reports performance on the held-out test split.

In [8]:
selected_prediction_file = experiment_dir / 'predictions' / 'location_aware_cnn_windows.csv'
selected = pd.read_csv(selected_prediction_file, dtype={'subject_id': str})
test_patients = aggregate_patients(selected[selected['split'].eq('test')], 'recording_mean')
test_metrics = metrics(test_patients)
pd.DataFrame([test_metrics]).to_json(experiment_dir / 'metrics' / 'location_aware_cnn_test.json', orient='records', indent=2)
test_metrics

{'n_subjects': 142,
 'accuracy': 0.8450704225352113,
 'macro_f1': np.float64(0.7258843890619592),
 'precision_present': np.float64(0.8888888888888888),
 'recall_present': np.float64(0.5925925925925926),
 'f1_present': np.float64(0.7111111111111111),
 'support_present': np.int64(27),
 'precision_absent': np.float64(0.8899082568807339),
 'recall_absent': np.float64(0.9238095238095239),
 'f1_absent': np.float64(0.9065420560747663),
 'support_absent': np.int64(105),
 'precision_unknown': np.float64(0.4666666666666667),
 'recall_unknown': np.float64(0.7),
 'f1_unknown': np.float64(0.56),
 'support_unknown': np.int64(10)}

## Results summary

The label audit identified 117 recordings from Present patients that were relabeled Absent because their recording locations did not contain a documented murmur. The location-aware CNN is selected by patient-level validation macro-F1 and evaluated on held-out patients by aggregating recording predictions.